In [1]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd


REGIONAL_FRACTION = 0.01
EXPECTED_COLUMNS = [
    "location",
    "timestamp",
    "method",
    "anomaly_score",
    "threshold",
    "is_anomaly",
]

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SEED_DIR = (
    REPO_ROOT
    / "outputs"
    / "pvgis_mtgflow"
    / "downstream_dense"
    / "seed_15"
)
TEST_CSV = SEED_DIR / "anomaly_scores.csv"
TRAIN_CSV = SEED_DIR / "train_anomaly_scores.csv"

for path in (TEST_CSV, TRAIN_CSV):
    if not path.is_file():
        raise FileNotFoundError(path)

print("Caricamento CSV MTGFlow...")
scores = pd.read_csv(TEST_CSV, parse_dates=["timestamp"])
train_scores = pd.read_csv(TRAIN_CSV, parse_dates=["timestamp"])
print(f"Score test caricati: {len(scores):,}")
print(f"Score training caricati: {len(train_scores):,}")


def analyse(frame, name):
    if list(frame.columns) != EXPECTED_COLUMNS:
        raise ValueError(f"{name}: schema inatteso: {list(frame.columns)}")
    if not pd.api.types.is_datetime64_any_dtype(frame["timestamp"]):
        frame["timestamp"] = pd.to_datetime(frame["timestamp"])
    if not pd.api.types.is_bool_dtype(frame["is_anomaly"]):
        raise TypeError(
            f"{name}: is_anomaly non booleano: {frame['is_anomaly'].dtype}"
        )

    location = (
        frame.groupby("location", sort=True)
        .agg(
            n_windows=("is_anomaly", "size"),
            n_anomaly=("is_anomaly", "sum"),
            mean_score=("anomaly_score", "mean"),
            threshold=("threshold", "first"),
            threshold_variants=("threshold", "nunique"),
        )
    )
    location["anomaly_rate"] = location["n_anomaly"] / location["n_windows"]

    regional = (
        frame.groupby("timestamp", sort=True)
        .agg(
            n_anomaly=("is_anomaly", "sum"),
            n_scored=("is_anomaly", "size"),
        )
    )
    regional["anomaly_fraction"] = regional["n_anomaly"] / regional["n_scored"]
    regional["is_rare"] = regional["anomaly_fraction"] >= REGIONAL_FRACTION

    rare = regional.loc[regional["is_rare"]].reset_index()
    if rare.empty:
        n_intervals = 0
        longest_interval = 0
    else:
        rare["_interval"] = (
            rare["timestamp"].diff().ne(pd.Timedelta(hours=1)).cumsum()
        )
        interval_lengths = rare.groupby("_interval").size()
        n_intervals = int(len(interval_lengths))
        longest_interval = int(interval_lengths.max())

    score_values = frame["anomaly_score"].to_numpy(dtype=np.float64, copy=False)
    threshold_values = frame["threshold"].to_numpy(dtype=np.float64, copy=False)
    flags = frame["is_anomaly"].to_numpy(dtype=bool, copy=False)
    score_quantiles = frame["anomaly_score"].quantile(
        [0, 0.01, 0.25, 0.5, 0.75, 0.99, 1]
    )
    rate_quantiles = location["anomaly_rate"].quantile(
        [0, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1]
    )

    report = {
        "name": name,
        "rows": int(len(frame)),
        "n_locations": int(frame["location"].nunique()),
        "n_timestamps": int(len(regional)),
        "timestamp_start": regional.index.min().isoformat(),
        "timestamp_end": regional.index.max().isoformat(),
        "years": sorted(int(year) for year in frame["timestamp"].dt.year.unique()),
        "methods": sorted(frame["method"].astype(str).unique().tolist()),
        "null_counts": {
            column: int(frame[column].isna().sum()) for column in frame.columns
        },
        "finite_scores": bool(np.isfinite(score_values).all()),
        "finite_thresholds": bool(np.isfinite(threshold_values).all()),
        "decision_mismatches": int(
            np.count_nonzero(flags != (score_values > threshold_values))
        ),
        "node_anomalies": int(flags.sum()),
        "node_anomaly_rate": float(flags.mean()),
        "windows_per_location_min": int(location["n_windows"].min()),
        "windows_per_location_max": int(location["n_windows"].max()),
        "nodes_per_timestamp_min": int(regional["n_scored"].min()),
        "nodes_per_timestamp_max": int(regional["n_scored"].max()),
        "threshold_variants_per_location_max": int(
            location["threshold_variants"].max()
        ),
        "required_nodes_for_rare_event": int(
            math.ceil(regional["n_scored"].max() * REGIONAL_FRACTION)
        ),
        "regional_rare_timestamps": int(regional["is_rare"].sum()),
        "regional_rare_rate": float(regional["is_rare"].mean()),
        "rare_intervals": n_intervals,
        "longest_rare_interval_hours": longest_interval,
        "max_simultaneous_anomalous_nodes": int(regional["n_anomaly"].max()),
        "max_regional_anomaly_fraction": float(
            regional["anomaly_fraction"].max()
        ),
        "score_quantiles": {
            str(q): float(value) for q, value in score_quantiles.items()
        },
        "location_anomaly_rate_quantiles": {
            str(q): float(value) for q, value in rate_quantiles.items()
        },
    }
    return report, location, regional


test_report, test_locations, regional_test = analyse(scores, "test_2019")
train_report, train_locations, regional_train = analyse(
    train_scores, "train_2016_2018"
)

common = test_locations.index.intersection(train_locations.index)
threshold_difference = np.abs(
    test_locations.loc[common, "threshold"].to_numpy()
    - train_locations.loc[common, "threshold"].to_numpy()
)
cross_split = {
    "same_locations": bool(
        set(test_locations.index.astype(str))
        == set(train_locations.index.astype(str))
    ),
    "common_locations": int(len(common)),
    "threshold_mismatch_locations": int(
        np.count_nonzero(threshold_difference > 1e-12)
    ),
    "max_threshold_difference": float(threshold_difference.max()),
}

print("\n=== AUDIT COMPLETO ===")
print(
    json.dumps(
        {"test": test_report, "train": train_report, "cross_split": cross_split},
        indent=2,
    )
)
print("\n=== 20 TIMESTAMP TEST PIÙ ANOMALI ===")
print(regional_test.nlargest(20, "n_anomaly").reset_index().to_string(index=False))
print("\n=== 20 TIMESTAMP TRAIN PIÙ ANOMALI ===")
print(regional_train.nlargest(20, "n_anomaly").reset_index().to_string(index=False))


Caricamento CSV MTGFlow...
Score test caricati: 9,997,449
Score training caricati: 30,223,296

=== AUDIT COMPLETO ===
{
  "test": {
    "name": "test_2019",
    "rows": 9997449,
    "n_locations": 1149,
    "n_timestamps": 8701,
    "timestamp_start": "2019-01-03T11:10:00",
    "timestamp_end": "2019-12-31T23:10:00",
    "years": [
      2019
    ],
    "methods": [
      "mtgflow"
    ],
    "null_counts": {
      "location": 0,
      "timestamp": 0,
      "method": 0,
      "anomaly_score": 0,
      "threshold": 0,
      "is_anomaly": 0
    },
    "finite_scores": true,
    "finite_thresholds": true,
    "decision_mismatches": 0,
    "node_anomalies": 194553,
    "node_anomaly_rate": 0.019460264313426356,
    "windows_per_location_min": 8701,
    "windows_per_location_max": 8701,
    "nodes_per_timestamp_min": 1149,
    "nodes_per_timestamp_max": 1149,
    "threshold_variants_per_location_max": 1,
    "required_nodes_for_rare_event": 12,
    "regional_rare_timestamps": 2579,
    "reg

In [2]:
print("fraction | train rare | test rare | test timestamps")

for fraction in [0.01, 0.02, 0.05, 0.10, 0.20]:
      train_mask = regional_train["anomaly_fraction"] >= fraction
      test_mask = regional_test["anomaly_fraction"] >= fraction

      print(
          f"{fraction:>8.0%} | "
          f"{train_mask.mean():>10.2%} | "
          f"{test_mask.mean():>9.2%} | "
          f"{int(test_mask.sum()):>15,}"
      )

fraction | train rare | test rare | test timestamps
      1% |     26.27% |    29.64% |           2,579
      2% |     17.93% |    20.77% |           1,807
      5% |      8.35% |    10.47% |             911
     10% |      3.76% |     5.07% |             441
     20% |      1.30% |     1.64% |             143
